In [ ]:
# ---------------------------------------------------------------
# Notebook 02: TON-IoT TRUE Multi-Class Baseline
# Log + Mutual Information + PCA + KMeansSMOTE
# Random Forest only
# ---------------------------------------------------------------

import pandas as pd
import numpy as np
from collections import Counter
import warnings
import time

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.decomposition import PCA

from imblearn.over_sampling import KMeansSMOTE

warnings.filterwarnings("ignore")

print("\nNotebook 02: TON-IoT TRUE Multi-Class Baseline | Log + MI + PCA + KMeansSMOTE + Random Forest\n")
start_time = time.time()

# ---------------------------------------------------------------
# PARAMETERS
# ---------------------------------------------------------------

N_SPLITS_CV = 3
RANDOM_STATE = 42
N_FEATURES_MI = 20
PCA_COMPONENTS = 0.95

TRAIN_PATH = "/kaggle/input/datasets/gowr1arun/unsw-nb15-and-ton-iot/toniot_network_train_80.csv"
TEST_PATH = "/kaggle/input/datasets/gowr1arun/unsw-nb15-and-ton-iot/toniot_network_test_20.csv"

TARGET_COL = "type"   # TRUE multi-class target

# ---------------------------------------------------------------
# 1. LOAD DATA
# ---------------------------------------------------------------

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f"Train 80%: {train_df.shape} | Test 20%: {test_df.shape}")

# ---------------------------------------------------------------
# 2. CLEAN + CONSISTENT ENCODE
# ---------------------------------------------------------------

def clean_and_encode_train_test(train_df, test_df):
    """
    Cleans train/test data consistently.

    Important:
    - Encoding is learned from train data only.
    - Unknown test categories are mapped to -1.
    - Target columns label/type are not transformed here.
    """

    leak_cols = [
        "src_ip",
        "dst_ip",
        "http_uri",
        "ssl_subject",
        "ssl_issuer",
        "timestamp",
        "date",
        "time",
    ]

    train = train_df.drop(
        columns=[c for c in leak_cols if c in train_df.columns],
        errors="ignore",
    ).copy()

    test = test_df.drop(
        columns=[c for c in leak_cols if c in test_df.columns],
        errors="ignore",
    ).copy()

    train.replace([np.inf, -np.inf], np.nan, inplace=True)
    test.replace([np.inf, -np.inf], np.nan, inplace=True)

    for col in train.columns:
        if col in ["label", "type"]:
            continue

        if train[col].dtype == "object":
            mode_value = train[col].mode()[0]

            train[col] = train[col].fillna(mode_value).astype(str)
            test[col] = test[col].fillna(mode_value).astype(str)

            categories = pd.Index(train[col].unique())
            mapping = {cat: idx for idx, cat in enumerate(categories)}

            train[col] = train[col].map(mapping).astype(int)
            test[col] = test[col].map(mapping).fillna(-1).astype(int)

        else:
            mean_value = train[col].mean()

            train[col] = train[col].fillna(mean_value)
            test[col] = test[col].fillna(mean_value)

    return train, test


train_df, test_df = clean_and_encode_train_test(train_df, test_df)

# ---------------------------------------------------------------
# 3. TRUE MULTI-CLASS TARGET SETUP
# ---------------------------------------------------------------

X_train_full = train_df.drop(columns=["label", "type"], errors="ignore")
X_test_full = test_df.drop(columns=["label", "type"], errors="ignore")

label_encoder = LabelEncoder()

y_train_full = label_encoder.fit_transform(train_df[TARGET_COL].astype(str))
y_test_full = label_encoder.transform(test_df[TARGET_COL].astype(str))

class_names = list(label_encoder.classes_)

print("\nTarget column:", TARGET_COL)
print("\nClass mapping:")
for idx, name in enumerate(class_names):
    print(f"{idx}: {name}")

print("\nClass distribution train:", Counter(y_train_full))
print("Class distribution test :", Counter(y_test_full))

# ---------------------------------------------------------------
# 4. LOG TRANSFORMATION
# ---------------------------------------------------------------

def log_transform(X):
    return np.log1p(np.abs(X))

# ---------------------------------------------------------------
# 5. DEFINE MODEL
# ---------------------------------------------------------------

rf_clf = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    n_jobs=-1,
)

# ---------------------------------------------------------------
# 6. CROSS-VALIDATION
# ---------------------------------------------------------------

cv = StratifiedKFold(
    n_splits=N_SPLITS_CV,
    shuffle=True,
    random_state=RANDOM_STATE,
)

cv_acc = []
cv_bal_acc = []
cv_prec = []
cv_rec = []
cv_f1_macro = []
cv_f1_weighted = []
cv_auc = []

for fold, (tr_idx, val_idx) in enumerate(cv.split(X_train_full, y_train_full), 1):
    print("\n" + "=" * 80)
    print(f"Fold {fold}")
    print("=" * 80)

    X_tr_raw = X_train_full.iloc[tr_idx]
    X_val_raw = X_train_full.iloc[val_idx]

    y_tr = y_train_full[tr_idx]
    y_val = y_train_full[val_idx]

    # 1. Log transform
    X_tr_log = log_transform(X_tr_raw)
    X_val_log = log_transform(X_val_raw)

    # 2. Scale
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr_log)
    X_val_scaled = scaler.transform(X_val_log)

    # 3. Mutual Information feature selection
    mi = mutual_info_classif(
        X_tr_scaled,
        y_tr,
        random_state=RANDOM_STATE,
    )

    top_idx = np.argsort(mi)[-N_FEATURES_MI:]

    X_tr_sel = X_tr_scaled[:, top_idx]
    X_val_sel = X_val_scaled[:, top_idx]

    # 4. PCA
    pca = PCA(
        n_components=PCA_COMPONENTS,
        random_state=RANDOM_STATE,
    )

    X_tr_pca = pca.fit_transform(X_tr_sel)
    X_val_pca = pca.transform(X_val_sel)

    # 5. KMeansSMOTE on training fold only
    sampler = KMeansSMOTE(
        random_state=RANDOM_STATE,
        k_neighbors=3,
        cluster_balance_threshold=0.01,
    )

    try:
        X_tr_bal, y_tr_bal = sampler.fit_resample(X_tr_pca, y_tr)
        print("KMeansSMOTE worked.")
    except Exception as e:
        print("KMeansSMOTE failed, continuing without resampling:", str(e)[:150])
        X_tr_bal, y_tr_bal = X_tr_pca, y_tr

    print("Before sampling:", Counter(y_tr))
    print("After sampling :", Counter(y_tr_bal))

    clf = rf_clf.__class__(**rf_clf.get_params())
    clf.fit(X_tr_bal, y_tr_bal)

    y_pred = clf.predict(X_val_pca)
    y_prob = clf.predict_proba(X_val_pca)

    fold_acc = accuracy_score(y_val, y_pred)
    fold_bal_acc = balanced_accuracy_score(y_val, y_pred)
    fold_prec = precision_score(y_val, y_pred, average="macro", zero_division=0)
    fold_rec = recall_score(y_val, y_pred, average="macro", zero_division=0)
    fold_f1_macro = f1_score(y_val, y_pred, average="macro", zero_division=0)
    fold_f1_weighted = f1_score(y_val, y_pred, average="weighted", zero_division=0)

    try:
        y_val_oh = pd.get_dummies(y_val)
        fold_auc = roc_auc_score(
            y_val_oh,
            y_prob,
            average="macro",
            multi_class="ovr",
        )
    except Exception:
        fold_auc = np.nan

    cv_acc.append(fold_acc)
    cv_bal_acc.append(fold_bal_acc)
    cv_prec.append(fold_prec)
    cv_rec.append(fold_rec)
    cv_f1_macro.append(fold_f1_macro)
    cv_f1_weighted.append(fold_f1_weighted)
    cv_auc.append(fold_auc)

    print(
        f"Fold {fold}: "
        f"Acc={fold_acc:.4f} | "
        f"BalAcc={fold_bal_acc:.4f} | "
        f"MacroPrec={fold_prec:.4f} | "
        f"MacroRec={fold_rec:.4f} | "
        f"MacroF1={fold_f1_macro:.4f} | "
        f"WeightedF1={fold_f1_weighted:.4f} | "
        f"AUC={fold_auc:.4f}"
    )

print("\n" + "=" * 80)
print("CV AVERAGE")
print("=" * 80)

print(f"Accuracy          : {np.mean(cv_acc):.4f}")
print(f"Balanced Accuracy : {np.mean(cv_bal_acc):.4f}")
print(f"Macro Precision   : {np.mean(cv_prec):.4f}")
print(f"Macro Recall      : {np.mean(cv_rec):.4f}")
print(f"Macro F1          : {np.mean(cv_f1_macro):.4f}")
print(f"Weighted F1       : {np.mean(cv_f1_weighted):.4f}")
print(f"Macro AUC OVR     : {np.nanmean(cv_auc):.4f}")

# ---------------------------------------------------------------
# 7. FINAL TRAIN + OFFICIAL TEST
# ---------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL TRAIN + OFFICIAL TEST")
print("=" * 80)

# 1. Log transform
X_train_log = log_transform(X_train_full)
X_test_log = log_transform(X_test_full)

# 2. Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_log)
X_test_scaled = scaler.transform(X_test_log)

# 3. Mutual Information feature selection
mi = mutual_info_classif(
    X_train_scaled,
    y_train_full,
    random_state=RANDOM_STATE,
)

top_idx = np.argsort(mi)[-N_FEATURES_MI:]

X_train_sel = X_train_scaled[:, top_idx]
X_test_sel = X_test_scaled[:, top_idx]

# 4. PCA
pca = PCA(
    n_components=PCA_COMPONENTS,
    random_state=RANDOM_STATE,
)

X_train_pca = pca.fit_transform(X_train_sel)
X_test_pca = pca.transform(X_test_sel)

print(f"Selected MI features : {N_FEATURES_MI}")
print(f"PCA output features  : {X_train_pca.shape[1]}")

# 5. KMeansSMOTE
sampler = KMeansSMOTE(
    random_state=RANDOM_STATE,
    k_neighbors=3,
    cluster_balance_threshold=0.01,
)

try:
    X_train_bal, y_train_bal = sampler.fit_resample(X_train_pca, y_train_full)
    print("KMeansSMOTE worked on full training set.")
except Exception as e:
    print("KMeansSMOTE failed on full training set, continuing without resampling:", str(e)[:150])
    X_train_bal, y_train_bal = X_train_pca, y_train_full

print("Before sampling:", Counter(y_train_full))
print("After sampling :", Counter(y_train_bal))

# 6. Train final RF
final_clf = rf_clf.__class__(**rf_clf.get_params())
final_clf.fit(X_train_bal, y_train_bal)

# 7. Predict official test
y_test_pred = final_clf.predict(X_test_pca)
y_test_prob = final_clf.predict_proba(X_test_pca)

# ---------------------------------------------------------------
# 8. TEST METRICS
# ---------------------------------------------------------------

test_acc = accuracy_score(y_test_full, y_test_pred)
test_bal_acc = balanced_accuracy_score(y_test_full, y_test_pred)
test_prec_macro = precision_score(y_test_full, y_test_pred, average="macro", zero_division=0)
test_rec_macro = recall_score(y_test_full, y_test_pred, average="macro", zero_division=0)
test_f1_macro = f1_score(y_test_full, y_test_pred, average="macro", zero_division=0)
test_f1_weighted = f1_score(y_test_full, y_test_pred, average="weighted", zero_division=0)

try:
    y_test_oh = pd.get_dummies(y_test_full)
    test_auc = roc_auc_score(
        y_test_oh,
        y_test_prob,
        average="macro",
        multi_class="ovr",
    )
except Exception:
    test_auc = np.nan

print("\n" + "=" * 80)
print("NOTEBOOK 02 TEST RESULTS: TRUE MULTI-CLASS RANDOM FOREST BASELINE")
print("=" * 80)

print(f"Accuracy          : {test_acc:.4f}")
print(f"Balanced Accuracy : {test_bal_acc:.4f}")
print(f"Macro Precision   : {test_prec_macro:.4f}")
print(f"Macro Recall      : {test_rec_macro:.4f}")
print(f"Macro F1          : {test_f1_macro:.4f}")
print(f"Weighted F1       : {test_f1_weighted:.4f}")
print(f"Macro AUC OVR     : {test_auc:.4f}")

print("\nPer-class report:")
print(
    classification_report(
        y_test_full,
        y_test_pred,
        target_names=class_names,
        digits=4,
        zero_division=0,
    )
)

print("\nConfusion matrix:")
cm = confusion_matrix(y_test_full, y_test_pred)
print(cm)

# ---------------------------------------------------------------
# 9. SAVE RESULTS
# ---------------------------------------------------------------

summary_df = pd.DataFrame([
    {
        "Model": "Random Forest",
        "Feature Strategy": "Log + Mutual Information + PCA",
        "Sampling": "KMeansSMOTE",
        "Task": "True multi-class using type column",
        "Accuracy": test_acc,
        "Balanced Accuracy": test_bal_acc,
        "Macro Precision": test_prec_macro,
        "Macro Recall": test_rec_macro,
        "Macro F1": test_f1_macro,
        "Weighted F1": test_f1_weighted,
        "Macro AUC OVR": test_auc,
        "MI Features": N_FEATURES_MI,
        "PCA Features": X_train_pca.shape[1],
    }
])

summary_path = "/kaggle/working/02_toniot_multiclass_baseline_summary.csv"
summary_df.to_csv(summary_path, index=False)

report_dict = classification_report(
    y_test_full,
    y_test_pred,
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)

report_df = pd.DataFrame(report_dict).transpose()

report_path = "/kaggle/working/02_toniot_multiclass_baseline_per_class_report.csv"
report_df.to_csv(report_path)

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{c}" for c in class_names],
    columns=[f"pred_{c}" for c in class_names],
)

cm_path = "/kaggle/working/02_toniot_multiclass_baseline_confusion_matrix.csv"
cm_df.to_csv(cm_path)

print("\nSaved files:")
print(summary_path)
print(report_path)
print(cm_path)

print(f"\nBenchmark complete in {time.time() - start_time:.2f} seconds.\n")

# ---------------------------------------------------------------
# 10. NOTEBOOK CONCLUSION
# ---------------------------------------------------------------

print("""
Notebook 02 conclusion:
This notebook is the true multi-class baseline because it uses the `type` column, not the binary `label` column.
It evaluates a Random Forest baseline with Log transformation, Mutual Information feature selection, PCA, and KMeansSMOTE.
The final model in later notebooks improves this baseline using rare-class targeted OVR LightGBM.
""")